In [ ]:
from pathlib import Path

import yaml
from transformers import AutoTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS = yaml.safe_load((ROOT / "configs" / "models.yaml").read_text())

# Three families: different templates, BOS conventions and padding defaults.
MODEL_NAMES = ["qwen3-0.6b", "gemma3-1b-it", "llama3.2-1b-it"]
toks = {name: AutoTokenizer.from_pretrained(MODELS[name]["hf_id"]) for name in MODEL_NAMES}

for name, tok in toks.items():
    print(f"{name:16s} bos={tok.bos_token!r:22s} padding_side={tok.padding_side}")

In [ ]:
def render(tok, user_content: str, template_kwargs: dict) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, **template_kwargs
    )

for name, tok in toks.items():
    print(f"===== {name}")
    print(repr(render(tok, "How do I bake bread?", MODELS[name]["template_kwargs"])))

In [ ]:
def show_tokens(tok, text: str) -> None:
    ids = tok(text, add_special_tokens=False)["input_ids"]   # list[int], length = seq
    for i, t in enumerate(ids):
        print(f"{i:3d}  {t:7d}  {tok.decode([t])!r}")

show_tokens(toks["gemma3-1b-it"], render(toks["gemma3-1b-it"], "How do I bake bread?", MODELS["gemma3-1b-it"]["template_kwargs"]))

In [ ]:
for name, tok in toks.items():
    text = render(tok, "How do I bake bread?", MODELS[name]["template_kwargs"])
    with_default = tok(text)["input_ids"]
    without = tok(text, add_special_tokens=False)["input_ids"]
    print(f"{name:16s} default={len(with_default):3d}  add_special_tokens=False={len(without):3d}  "
          f"first two tokens (default): {[tok.decode([t]) for t in with_default[:2]]}")

In [ ]:
tok = toks["qwen3-0.6b"]
enc = tok("Hi there?", add_special_tokens=False, return_offsets_mapping=True)
for tid, (s, e) in zip(enc["input_ids"], enc["offset_mapping"]):
    print(f"{tok.decode([tid])!r:10s} chars [{s}, {e})  -> {'Hi there?'[s:e]!r}")

In [ ]:
def find_positions(tok, user_content: str, template_kwargs: dict) -> dict:
    """Render a single-turn chat prompt and locate the instruction and post-instruction tokens.

    Returns:
        text:        rendered prompt string
        input_ids:   list[int], tokenized with add_special_tokens=False (template already has BOS)
        inst_start:  index of the first instruction token
        t_inst:      index of the last instruction token
        t_post_inst: list of indices after t_inst (end-of-turn + assistant header), to the end
    """
    text = render(tok, user_content, template_kwargs)

    # 1. character span of the instruction inside the rendered string
    start = text.rfind(user_content)
    if start == -1:
        # e.g. the Gemma/Llama templates strip surrounding whitespace -> strip prompts in data prep
        raise ValueError(f"user content not found verbatim in rendered template: {user_content!r}")
    end = start + len(user_content)

    # 2. tokenize, keeping each token's (char_start, char_end)
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    ids, offsets = enc["input_ids"], enc["offset_mapping"]   # both length seq

    # 3. characters -> tokens
    inst_start = min(i for i, (s, e) in enumerate(offsets) if e > start)
    t_inst = max(i for i, (s, e) in enumerate(offsets) if s < end)

    # 4. refuse to guess if a token mixes instruction and template text
    if offsets[t_inst][1] > end:
        raise ValueError(f"token {t_inst} straddles the instruction end: {tok.decode([ids[t_inst]])!r}")

    return {
        "text": text,
        "input_ids": ids,
        "inst_start": inst_start,
        "t_inst": t_inst,
        "t_post_inst": list(range(t_inst + 1, len(ids))),
    }

In [ ]:
PROMPTS = {
    "eng_Latn": "How do I bake bread?",
    "npi_Deva": "रोटी कसरी बनाउने?",
    "npi_Latn": "roti kasari banaune?",
    "rus_Cyrl": "Как испечь хлеб?",
    "zho_Hans": "如何烤面包？",
}

for name, tok in toks.items():
    kwargs = MODELS[name]["template_kwargs"]
    n_post = set()
    print(f"===== {name}")
    for lang, prompt in PROMPTS.items():
        p = find_positions(tok, prompt, kwargs)
        ids, t_inst, post = p["input_ids"], p["t_inst"], p["t_post_inst"]

        assert tok.decode([ids[t_inst]]).endswith(prompt[-1]), (lang, tok.decode([ids[t_inst]]))
        assert p["text"].endswith(tok.decode([ids[i] for i in post])), lang
        assert tok.decode(ids[p["inst_start"] : t_inst + 1]).strip() == prompt, lang
        if tok.bos_token_id is not None:
            assert ids[0] == tok.bos_token_id and ids[1] != tok.bos_token_id, "BOS missing or doubled"
        n_post.add(len(post))

        n_inst = t_inst - p["inst_start"] + 1
        print(f"  {lang}: instruction tokens={n_inst:3d}  t_inst={t_inst:3d}  n_post_inst={len(post)}")
    assert len(n_post) == 1, f"post-instruction length varies across languages: {n_post}"
print("all checks passed")

In [ ]:
import torch

def pad_batch(seqs: list[list[int]], pad_id: int, side: str):
    """Pad token-id lists into a rectangle.
    Returns input_ids [B, T], attention_mask [B, T], shift [B] (how far each row's tokens moved)."""

    B, T = len(seqs), max(len(s) for s in seqs)
    input_ids = torch.full((B,T), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((B,T), dtype=torch.long)
    shift = torch.zeros(B, dtype=torch.long)
    for b, s in enumerate(seqs):
        n = len(s)
        if side == "right":
            input_ids[b, :n] = torch.tensor(s)
            attention_mask[b, :n] = 1
        else:
            input_ids[b, T - n:] = torch.tensor(s)
            attention_mask[b, T - n:] = 1
            shift[b] = T - n
    return input_ids, attention_mask, shift

In [ ]:
name = "qwen3-0.6b"
tok, kw = toks[name], MODELS[name]["template_kwargs"]
prompts = ["How do I bake bread?", "रोटी कसरी बनाउने?", "Hi"]

ps = [find_positions(tok, p, kw) for p in prompts]
seqs = [p["input_ids"] for p in ps]
lens = torch.tensor([len(s) for s in seqs])
t_inst = torch.tensor([p["t_inst"] for p in ps])

pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
for side in ["right", "left"]:
    ids, mask, shift = pad_batch(seqs, pad_id, side)
    t_inst_pad = t_inst + shift
    last_pad = shift + lens - 1
    print(side, ids.shape, "t_inst:", t_inst_pad.tolist(), "last:", last_pad.tolist())
    for b in range(len(seqs)):
        assert ids[b, t_inst_pad[b]] == seqs[b][t_inst[b]]

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODELS[name]["hf_id"], dtype=torch.float32).eval()

with torch.inference_mode():
    ref_inst, ref_last = [], []
    for s, p in zip(seqs, ps):
        out = model(torch.tensor([s]), output_hidden_states=True)
        hs = torch.stack(out.hidden_states)
        ref_inst.append(hs[:, 0, p["t_inst"]])
        ref_last.append(hs[:, 0, -1])
        
    for side in ["right", "left"]:
        ids, mask, shift = pad_batch(seqs, pad_id, side)
        out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
        hs = torch.stack(out.hidden_states)                  # [L+1, B, T, d_model]

        B = ids.shape[0]
        rows = torch.arange(B)                               # [B]: 0, 1, 2
        acts_inst = hs[:, rows, t_inst + shift]              # [L+1, B, d_model]
        acts_last = hs[:, rows, shift + lens - 1]            # [L+1, B, d_model]

        for b in range(B):
            d1 = (acts_inst[:, b] - ref_inst[b]).abs().max().item()
            d2 = (acts_last[:, b] - ref_last[b]).abs().max().item()
            print(f"{side:5s} prompt {b}: max|diff| t_inst={d1:.1e}  last={d2:.1e}")

### Now test the same thing for TransformerBridge ..(we tested things up untill now using residuals from hf calls)

In [ ]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(MODELS[name]["hf_id"], device="cpu", dtype=torch.float32)
bridge.eval()
L, d = bridge.cfg.n_layers, bridge.cfg.d_model

p = ps[0]
ids = torch.tensor([p["input_ids"]])
names = [f"blocks.{l}.hook_resid_post" for l in range(L)]

with torch.inference_mode():
    _, cache = bridge.run_with_cache(ids, names_filter=names)
    hs = torch.stack(model(ids, output_hidden_states=True).hidden_states)[:, 0]

for l in [0, L // 2, L - 2]:
    print(l, (cache[names[l]][0] - hs[l + 1]).abs().max().item())  

last_tl = cache[names[L - 1]][0]
print("last, raw   :", (last_tl - hs[L]).abs().max().item())
print("last, normed:", (model.model.norm(last_tl) - hs[L]).abs().max().item())

In [ ]:
def collect_resid(bridge, ps: list[dict], pad_id: int, side: str = "left"):
    """Residual stream after every block, at t_inst and at every t_post_inst token.
    Returns acts_inst [B, L, d], acts_post [B, P, L, d] (float32, CPU)."""
    seqs = [p["input_ids"] for p in ps]
    ids, mask, shift = pad_batch(seqs, pad_id, side)
    device = bridge.cfg.device
    names = [f"blocks.{l}.hook_resid_post" for l in range(bridge.cfg.n_layers)]

    with torch.inference_mode():
        _, cache = bridge.run_with_cache(ids.to(device), attention_mask=mask.to(device), names_filter=names)
        resid = torch.stack([cache[n] for n in names], dim=2)
        rows = torch.arange(len(ps))
        t_inst = torch.tensor([p["t_inst"] for p in ps]) + shift
        t_post = torch.tensor([p["t_post_inst"] for p in ps]) + shift[:, None]

        acts_inst = resid[rows, t_inst]
        acts_post = resid[rows[:, None], t_post]
        return acts_inst.float().cpu(), acts_post.float().cpu()

In [ ]:
for side in ["left", "right"]:
    acts_inst, acts_post = collect_resid(bridge, ps, pad_id, side)
    print(side, acts_inst.shape, acts_post.shape)
    for b, p in enumerate(ps):
        with torch.inference_mode():
            hs = torch.stack(model(torch.tensor([p["input_ids"]]), output_hidden_states=True).hidden_states)
        ref = hs[1:L, 0]                                              # [L-1, T, d]: blocks 0..L-2 (skip embeddings and normed last)
        ref_inst = ref[:, p["t_inst"]]                                # [L-1, d]
        ref_post = ref[:, p["t_post_inst"]].transpose(0, 1)           # [L-1, P, d] -> [P, L-1, d]
        ok = torch.allclose(acts_inst[b, :L-1], ref_inst, atol=1e-3, rtol=1e-4) and \
             torch.allclose(acts_post[b, :, :L-1], ref_post, atol=1e-3, rtol=1e-4)
        print(f"  prompt {b}: max|diff| inst={(acts_inst[b, :L-1] - ref_inst).abs().max():.1e}  "
              f"post={(acts_post[b, :, :L-1] - ref_post).abs().max():.1e}  allclose={ok}")